# Week 41, Day 6: Forge: Week 41 challenge, quiz and deep read

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w041/d6.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

In [ ]:
print("1.", 1.875 * 0.4 / 0.8, "m/s")
print("2.", round(0.05 / (1 - 0.98 + 0.05), 3))
print("3. u = -q x / (q + r) = -x / 2")

### Exercise: Part A: fitting the two-rate model

Write loss(p): return a large number unless 0 < A_f < A_s < 1 and 0 < B_s < B_f < 1, otherwise the mean squared error between simulate(p) and y on the training trials. Minimisation from three starting points is done for you.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
r = np.random.default_rng(41)
true = (0.6, 0.2, 0.99, 0.03)                                  # A_f, B_f, A_s, B_s
nA, nD, nC = 200, 10, 100
f = np.r_[np.ones(nA), -np.ones(nD), np.zeros(nC)]
clamp = np.r_[np.zeros(nA + nD, bool), np.ones(nC, bool)]

def simulate(p, noise=0.0, g=None):
    Af, Bf, As, Bs = p; xf = xs = 0.0; out = []
    for n in range(len(f)):
        x = xf + xs; out.append(x + (noise * g.normal() if g is not None else 0.0))
        e = 0.0 if clamp[n] else f[n] - x
        xf = Af * xf + Bf * e; xs = As * xs + Bs * e
    return np.array(out)

y = simulate(true, noise=0.08, g=r)
train, te = np.arange(nA + nD), np.arange(nA + nD, len(f))

def loss(p):
    ...

best = None
for start in [(0.5, 0.3, 0.98, 0.05), (0.7, 0.15, 0.995, 0.01), (0.4, 0.4, 0.97, 0.08)]:
    res = minimize(loss, start, method="Nelder-Mead", options={"maxiter": 3000, "xatol": 1e-6, "fatol": 1e-9})
    if best is None or res.fun < best.fun:
        best = res
p_fit = best.x
pred = simulate(p_fit)
mse_fit = np.mean((pred[te] - y[te]) ** 2); mse_true = np.mean((simulate(true)[te] - y[te]) ** 2)
print("fitted A_f, B_f, A_s, B_s:", np.round(p_fit, 3), " true:", true)
print(f"clamp prediction error {mse_fit:.4f} (true model {mse_true:.4f})")
plt.plot(y, ".", ms=3, alpha=0.5, label="participant"); plt.plot(pred, label="fitted model"); plt.axvline(nA + nD, color="k", ls=":")
plt.xlabel("trial"); plt.ylabel("output"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_params():
    "the fitted slow retention is within 0.01 of the true 0.99 and the fast retention within 0.15 of 0.6"
    assert abs(p_fit[2] - 0.99) < 0.01 and abs(p_fit[0] - 0.6) < 0.15

def test_prediction():
    "the predicted clamp trials have a mean squared error within 10% of the true model's, and the prediction rebounds above 0.2"
    assert mse_fit < 1.1 * mse_true and pred[te].max() > 0.2

test_params()
test_prediction()
print("All checks passed")

### Exercise: Part B: estimation plus control

Complete the Kalman filter inside run(): after the controller acts and the arm moves, predict the estimate (xh = A xh + B u, P = A P Aᵀ + W), then update it with the noisy position measurement y using the gain K = P Hᵀ / (H P Hᵀ + s²).

In [ ]:
dt, T = 0.01, 60
A = np.array([[1, dt], [0, 1]]); B = np.array([[0], [dt]]); Qf = np.diag([1e4, 1e2]); Rc = np.array([[1e-4]])
S = Qf.copy(); Ks = []
for t in range(T):
    K = np.linalg.solve(Rc + B.T @ S @ B, B.T @ S @ A); Ks.append(K); S = A.T @ S @ (A - B @ K)
Ks = Ks[::-1]
H = np.array([[1.0, 0.0]]); W = np.diag([1e-6, 4e-4]); w_sd = np.sqrt(np.diag(W))
x = np.array([-0.1, 0.0]); planned = []
for t in range(T):
    u = (-Ks[t] @ x)[0]; planned.append(u); x = A @ x + B[:, 0] * u
g = np.random.default_rng(2002)

def run(mode, s_obs=0.0, trials=200):
    ends = []
    for _ in range(trials):
        x = np.array([-0.1, 0.0]); xh = x.copy(); P = np.zeros((2, 2))
        for t in range(T):
            u = planned[t] if mode == "open" else (-Ks[t] @ (x if mode == "full" else xh))[0]
            x = A @ x + B[:, 0] * u + w_sd * g.normal(size=2)
            if mode == "lqg":
                ...
        ends.append(x[0])
    return 1000 * np.std(ends)

sd = {"full state": run("full"), "LQG 2 mm": run("lqg", 0.002), "LQG 10 mm": run("lqg", 0.01), "LQG 30 mm": run("lqg", 0.03), "open loop": run("open")}
print({k: f"{v:.1f} mm" for k, v in sd.items()}, "end-point standard deviation")
plt.bar(list(sd), list(sd.values())); plt.xticks(rotation=30); plt.ylabel("end-point SD (mm)"); plt.tight_layout(); plt.show()

In [ ]:
import numpy as np
def test_order():
    "end-point spread rises with sensory noise, from the full-state controller to open loop"
    v = [sd["full state"], sd["LQG 2 mm"], sd["LQG 10 mm"], sd["LQG 30 mm"], sd["open loop"]]
    assert all(a < b for a, b in zip(v, v[1:]))

def test_feedback_helps():
    "even with 30 mm sensory noise, feedback beats open loop by more than a factor of 3"
    assert sd["open loop"] > 3 * sd["LQG 30 mm"]

test_order()
test_feedback_helps()
print("All checks passed")